# 05 — Final Load Prep: KPI Computation & Tableau Export

This notebook computes all 7 KPIs defined in the project framework, creates a Tableau-ready dataset with clean column names and pre-computed metrics, and exports it for dashboard building.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_olist_dataset.csv"

df = pd.read_csv(DATA_PATH, parse_dates=[
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
])
print(f"Loaded: {len(df):,} rows × {df.columns.size} columns")

---
## 1. Order-Level KPI Table

One row per order with all KPIs pre-computed. This is the primary file for Tableau.

In [ ]:
order_kpi = df.groupby("order_id").agg(
    customer_unique_id=("customer_unique_id", "first"),
    customer_state=("customer_state", "first"),
    customer_city=("customer_city", "first"),
    purchase_date=("order_purchase_timestamp", "first"),
    purchase_month=("purchase_month", "first"),
    purchase_year=("purchase_year", "first"),
    purchase_month_num=("purchase_month_num", "first"),
    purchase_weekday=("purchase_weekday", "first"),
    purchase_hour=("purchase_hour", "first"),
    order_item_count=("order_item_id", "max"),
    total_items_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    total_order_value=("total_item_value", "sum"),
    total_payment=("total_payment_value", "first"),
    primary_payment_type=("primary_payment_type", "first"),
    max_installments=("max_installments", "first"),
    product_category_main=("product_category", lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else "mixed"),
    review_score=("review_score", "first"),
    actual_delivery_days=("actual_delivery_days", "first"),
    delivery_delay_days=("delivery_delay_days", "first"),
    is_on_time=("is_on_time", "first"),
    seller_state=("seller_state", "first"),
).reset_index()

print(f"Order KPI table: {len(order_kpi):,} rows × {order_kpi.columns.size} columns")

## 2. KPI Computation — All 7 Metrics

In [ ]:
# --- KPI 1: Monthly Revenue ---
monthly_revenue = order_kpi.groupby("purchase_month")["total_payment"].sum().reset_index()
monthly_revenue.columns = ["month", "revenue_brl"]
monthly_revenue = monthly_revenue[monthly_revenue["month"] >= "2017-01"]

# --- KPI 2: Average Order Value ---
aov = order_kpi["total_payment"].mean()
print(f"KPI 2 — Average Order Value:     BRL {aov:,.2f}")

# --- KPI 3: On-Time Delivery Rate ---
ot_rate = order_kpi["is_on_time"].mean()
print(f"KPI 3 — On-Time Delivery Rate:    {ot_rate:.1%}")

# --- KPI 4: Customer Satisfaction (CSAT) ---
csat = order_kpi["review_score"].mean()
print(f"KPI 4 — Avg Review Score (CSAT):  {csat:.2f} / 5")

# --- KPI 5: Repeat Purchase Rate ---
cust_orders = order_kpi.groupby("customer_unique_id").size()
repeat_rate = (cust_orders > 1).mean()
print(f"KPI 5 — Repeat Purchase Rate:     {repeat_rate:.1%}")

# --- KPI 6: Avg Delivery Delay ---
avg_delay = order_kpi["delivery_delay_days"].dropna().mean()
print(f"KPI 6 — Avg Delivery Delay:       {avg_delay:.1f} days (negative = early)")

# --- KPI 7: Freight-to-Price Ratio ---
avg_freight_ratio = order_kpi["total_freight"].sum() / order_kpi["total_items_price"].replace(0, np.nan).sum()
print(f"KPI 7 — Platform Freight Ratio:   {avg_freight_ratio:.1%}")

## 3. Segmented KPI Tables for Tableau

Pre-aggregated tables that make Tableau filter/parameter construction easier.

In [ ]:
# --- State-level KPIs ---
state_kpi = order_kpi.groupby("customer_state").agg(
    total_revenue=("total_payment", "sum"),
    order_count=("order_id", "count"),
    avg_order_value=("total_payment", "mean"),
    avg_review_score=("review_score", "mean"),
    on_time_rate=("is_on_time", "mean"),
    avg_delivery_days=("actual_delivery_days", "mean"),
    avg_delivery_delay=("delivery_delay_days", "mean"),
).reset_index().round(2)

print(f"State KPI table: {len(state_kpi)} states\n")
state_kpi.sort_values("total_revenue", ascending=False).head(10)

In [ ]:
# --- Category-level KPIs ---
cat_kpi = order_kpi.groupby("product_category_main").agg(
    total_revenue=("total_payment", "sum"),
    order_count=("order_id", "count"),
    avg_order_value=("total_payment", "mean"),
    avg_review_score=("review_score", "mean"),
    on_time_rate=("is_on_time", "mean"),
).reset_index().round(2)

print(f"Category KPI table: {len(cat_kpi)} categories\n")
cat_kpi.sort_values("total_revenue", ascending=False).head(10)

In [ ]:
# --- Monthly trend KPIs ---
monthly_kpi = order_kpi.groupby("purchase_month").agg(
    revenue=("total_payment", "sum"),
    orders=("order_id", "count"),
    aov=("total_payment", "mean"),
    avg_review=("review_score", "mean"),
    on_time_rate=("is_on_time", "mean"),
    avg_delivery_days=("actual_delivery_days", "mean"),
).reset_index().round(2)
monthly_kpi = monthly_kpi[monthly_kpi["purchase_month"] >= "2017-01"]

print(f"Monthly KPI table: {len(monthly_kpi)} months")
monthly_kpi.tail(6)

In [ ]:
# --- Payment type KPIs ---
pay_kpi = order_kpi.groupby("primary_payment_type").agg(
    total_revenue=("total_payment", "sum"),
    order_count=("order_id", "count"),
    avg_order_value=("total_payment", "mean"),
    avg_review=("review_score", "mean"),
    avg_installments=("max_installments", "mean"),
).reset_index().round(2)

pay_kpi

---
## 4. Export All Tableau-Ready Files

In [ ]:
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

exports = {
    "tableau_orders_kpi.csv": order_kpi,
    "tableau_state_kpi.csv": state_kpi,
    "tableau_category_kpi.csv": cat_kpi,
    "tableau_monthly_kpi.csv": monthly_kpi,
    "tableau_payment_kpi.csv": pay_kpi,
}

for filename, data in exports.items():
    path = output_dir / filename
    data.to_csv(path, index=False)
    print(f"  ✓ {filename:35s} {len(data):>8,} rows  ({path.stat().st_size / 1024:.0f} KB)")

print(f"\nAll Tableau-ready files exported to {output_dir}")

## 5. KPI Summary Card

Quick reference for the dashboard executive view.

In [ ]:
print("=" * 55)
print("  OLIST MARKETPLACE — KPI SCORECARD")
print("=" * 55)
print(f"  Total Revenue:          BRL {order_kpi['total_payment'].sum():>14,.2f}")
print(f"  Total Orders:           {len(order_kpi):>14,}")
print(f"  Unique Customers:       {order_kpi['customer_unique_id'].nunique():>14,}")
print(f"  Unique Categories:      {order_kpi['product_category_main'].nunique():>14,}")
print(f"  Date Range:             2017-01 → 2018-08")
print("-" * 55)
print(f"  Avg Order Value:        BRL {aov:>14,.2f}")
print(f"  CSAT (Avg Review):      {csat:>14.2f} / 5")
print(f"  On-Time Delivery:       {ot_rate:>13.1%}")
print(f"  Avg Delivery Days:      {order_kpi['actual_delivery_days'].mean():>14.1f}")
print(f"  Avg Delivery Delay:     {avg_delay:>14.1f} days")
print(f"  Repeat Purchase Rate:   {repeat_rate:>13.1%}")
print(f"  Platform Freight Ratio: {avg_freight_ratio:>13.1%}")
print("=" * 55)